# Generate-then-Refine con más datos sintéticos (R6) — más oraciones por categoría, dentro de cada fold

Repite Generate-then-Refine dentro de cada fold pidiéndole **más oraciones por categoría** a Claude (`--cantidad`). Se piden en lotes de 20 (una llamada por lote);
cada lote nuevo usa otra ventana de ejemplos few-shot, para que no repita las oraciones del anterior, y se descartan las repetidas y las copias de oraciones reales.
Todo lo hace `validacion_cruzada_en_linea.py`, con el filtro de marcador que usa solo lo que afirman las fuentes (`marcadores_fuentes.csv`).

**Estrategia (la misma que el entrenamiento): test / pool / core / dev.** Para cada fold: *test* = el fold (no se toca);
*pool* = los otros 4 folds; el pool se parte en *dev interno* (~1/6) y *core*. Hay dos etapas:
- **Etapa core (elegir `C`):** el aumento se genera/filtra solo con el core y se mide en el dev interno.
- **Etapa pool (modelo final):** el aumento se genera/filtra con todo el pool y se mide en el test.

Así, el dev interno nunca influye en el aumento con el que se elige `C`. En todas las etapas se descartan las filas sintéticas
idénticas a una oración del test.

**Cuánto aumentar.** Con 20 por categoría, Generate-then-Refine aporta ~17% del tamaño del pool real (~95 oraciones aprobadas por etapa), muy por debajo de
Mixup (100%) y Retrotraducción (85%). Recomendación: probar **×2 y ×4**.

| `NIVELES` (por categoría y etapa) | Aprobadas aprox. por etapa | % del pool real | Costo extra aprox. |
|---|---|---|---|
| 20 (ya hecho) | ~95 | ~17% | — |
| **40** | ~190 | ~34% | ~US$0.5 |
| **80** | ~380 | ~68% | ~US$1 más |
| 120 (opcional) | ~570 | ~100% (como Mixup) | ~US$1 más |

El lote 0 de cada fold ya está en tu Drive de la corrida anterior y se **reutiliza gratis**; cada nivel solo paga los lotes que faltan, y el nivel 80 reutiliza
los del 40. Más allá de ~100% el sintético supera al real y el riesgo de dominarlo crece. Estimación de costo: ~US$0.5 por cada 20 por categoría con
`claude-sonnet-4-6` (US$3/US$15 por millón de tokens de entrada/salida), verifícala en tu consola de Anthropic.

**Antes de correr:**
1. `git commit` + `git push` del repo con `validacion_cruzada_en_linea.py` y `marcadores_fuentes.csv` actualizados. Esta Colab clona tu repo.
2. En Colab: ícono de llave (**Secretos**) → `ANTHROPIC_API_KEY` con acceso desde el cuaderno. No hace falta GPU.

**Resultado:** un solo `.zip` con lo generado de todos los niveles y, por cada nivel, predicciones, resumen, comparaciones pareadas, curvas ROC (CSV, grilla e
imagen por experimento) y el volumen sintético usado. Cada archivo generado se guarda en Drive apenas se produce: si algo falla, vuelve a correr el paso 4 y **no se
vuelve a pagar** lo ya generado.

## 0. Montar Google Drive (aquí queda guardado lo generado)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repo

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git log --oneline -3

ruta = '/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py'
assert os.path.exists(ruta), ('No existe validacion_cruzada_en_linea.py: haz git commit + git push del repo reorganizado '
                              '(carpetas oe2_...) con los cambios nuevos y vuelve a correr esta celda.')
print('OK')

## 2. Instalar dependencias

In [ ]:
!pip install -q anthropic python-dotenv sentence-transformers scikit-learn
import torch
print('GPU disponible:', torch.cuda.is_available())

## 3. API key (desde los Secretos de Colab) y prueba de conexión

La clave nunca se imprime. La prueba usa 10 tokens de salida.

In [ ]:
import os
from google.colab import userdata
import anthropic

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
cliente = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
r = cliente.messages.create(model="claude-sonnet-4-6", max_tokens=10, messages=[{"role": "user", "content": "Responde solo: ok"}])
print("Conexion OK:", r.content[0].text)

## 4. Generar y evaluar, por nivel de volumen (todo en una corrida)

`NIVELES` = oraciones pedidas por categoría y etapa (múltiplos de 20). Para cada nivel y fold: reutiliza los lotes ya guardados, pide a Claude solo los que faltan
(core y pool; cada archivo se guarda en Drive), une los lotes quitando repetidas y copias, se queda con las aprobadas por los filtros que no copian el test,
extrae embeddings de los 3 modelos y entrena.

**Llamadas a la API:** por nivel y fold, (lotes nuevos) × 7 categorías × 2 etapas. Con `[40, 80]` y el lote 0 ya hecho: 5 × 7 × 2 × (1 + 2) = **210 llamadas**.

In [ ]:
import os, subprocess

def correr(comando, cwd="/content/shiwilu-tesis"):
    """Corre un comando mostrando la salida en vivo; se detiene si falla."""
    proc = subprocess.Popen(
        comando, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"},
    )
    for linea in proc.stdout:
        print(linea, end="")
    proc.wait()
    assert proc.returncode == 0, f"Fallo (codigo {proc.returncode}): {' '.join(comando)}"

NIVELES = [40, 80, 120]   # oraciones por categoria y etapa (20 ya está hecho)
PRUEBA = False       # True = prueba rapida y barata: solo el fold 0, con cache y etiquetas aparte (no contamina la corrida completa)
CACHE = "/content/drive/MyDrive/shiwilu_en_linea" + ("_prueba" if PRUEBA else "")
SUFIJO = "_prueba" if PRUEBA else ""
TAGS = [f"en_linea_generate_then_refine_colab_c{n}{SUFIJO}" for n in NIVELES]

for cantidad in NIVELES:
    print(f"\n=============== Generate-then-Refine, {cantidad} por categoría y etapa ===============")
    correr(["python", "oe2_aumento_de_datos/evaluacion/validacion_cruzada_en_linea.py",
            "--tecnica", "generate_then_refine", "--cantidad", str(cantidad),
            "--cache", CACHE, "--etiqueta", f"colab_c{cantidad}{SUFIJO}"] + (["--folds", "0"] if PRUEBA else []))

## 5. Tablas y curvas ROC (las mismas que el repo), por nivel

In [ ]:
if not PRUEBA:
    import pandas as pd
    from IPython.display import display, Image
    RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"

    for TAG in TAGS:
        ENTRADA = f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_{TAG}.csv"
        print("=" * 20, TAG, "=" * 20)
        correr(["python", "oe2_aumento_de_datos/evaluacion/comparacion_pareada.py", "--entrada", ENTRADA, "--etiqueta", TAG])
        correr(["python", "oe2_aumento_de_datos/evaluacion/curvas_roc.py", "--entrada", ENTRADA, "--etiqueta", TAG])
        display(Image(f"{RES}/curvas_roc_{TAG}.png"))
        auc_macro = pd.read_csv(f"{RES}/curvas_roc_{TAG}.csv").query("categoria == 'macro'")
        display(auc_macro.pivot(index="modelo", columns="tecnica", values="auc").round(4))

if PRUEBA:
    print('Modo prueba: solo se evaluo el fold 0; las tablas y curvas ROC necesitan los 5 folds.')

## 6. Volumen usado y comparación frente a `sin_aumento` y a Generate-then-Refine vigente

In [ ]:
if not PRUEBA:
    import sys
    import numpy as np
    sys.path.insert(0, "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion")
    from comparacion_pareada import diferencia_pareada

    P_vig = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion.csv")
    vigente = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion.csv")
    def f1(tabla, tecnica):
        return tabla[tabla.tecnica == tecnica].set_index("modelo").f1_macro_agrupado

    columnas = {"sin aumento": f1(vigente, "sin_aumento"), "GtR vigente (20, filtro original)": f1(vigente, "generate_then_refine")}
    pcts = {}
    for nivel, TAG in zip(NIVELES, TAGS):
        r = pd.read_csv(f"{RES}/validacion_cruzada_resumen_sin_puntuacion_{TAG}.csv")
        columnas[f"GtR {nivel}"] = f1(r, "generate_then_refine")
        v = pd.read_csv(f"{RES}/volumen_sintetico_sin_puntuacion_{TAG}.csv")
        pcts[nivel] = round(v[v.etapa == "pool"].pct_de_los_reales.mean(), 1)
    display(pd.DataFrame(columnas).round(4))
    print("Volumen sintetico usado en la etapa pool, como % de las oraciones reales del pool:", pcts)

    rng = np.random.default_rng(42)
    filas = []
    for nivel, TAG in zip(NIVELES, TAGS):
        P_nuevo = pd.read_csv(f"{RES}/validacion_cruzada_predicciones_sin_puntuacion_{TAG}.csv")
        P = pd.concat([P_nuevo[P_nuevo.tecnica == "generate_then_refine"].assign(tecnica="nuevo"),
                       P_vig[P_vig.tecnica == "sin_aumento"],
                       P_vig[P_vig.tecnica == "generate_then_refine"].assign(tecnica="vigente")])
        for modelo in ["labse", "mbert", "xlmr"]:
            for a, b in [("nuevo", "sin_aumento"), ("nuevo", "vigente")]:
                d, lo, hi = diferencia_pareada(P, (modelo, a), (modelo, b), rng)
                filas.append({"nivel": nivel, "modelo": modelo, "comparacion": f"{a} - {b}", "diferencia_f1": round(d, 4),
                              "ic95_bajo": round(lo, 4), "ic95_alto": round(hi, 4), "distinguible_de_cero": not (lo <= 0 <= hi)})
    display(pd.DataFrame(filas))

## 7. Descargar el resultado (un solo `.zip`)

Contiene `en_linea/generate_then_refine/` (lotes generados, todos los niveles) y `resultados/` (por nivel: predicciones y resumen con los 12 experimentos,
comparaciones pareadas, curvas ROC en CSV, grilla e imágenes por experimento, y el volumen sintético). En tu computadora: lo generado va en
`oe2_aumento_de_datos/tecnicas_aumento/salidas/en_linea/` y `resultados/` en `oe2_aumento_de_datos/evaluacion/resultados/`. Lo generado también queda en Drive.

In [ ]:
import glob, shutil
from google.colab import files

RES = "/content/shiwilu-tesis/oe2_aumento_de_datos/evaluacion/resultados"
carpeta = "/content/salida_gtr_en_linea"
shutil.rmtree(carpeta, ignore_errors=True)
os.makedirs(carpeta + "/resultados"); os.makedirs(carpeta + "/en_linea")
shutil.copytree(f"{CACHE}/generate_then_refine", carpeta + "/en_linea/generate_then_refine")      # lo generado, todos los niveles
for ruta_ in glob.glob(f"{RES}/*en_linea_generate_then_refine_colab_c*"):                             # CSV, PNG y carpetas de curvas ROC de todos los niveles
    destino = carpeta + "/resultados/" + os.path.basename(ruta_)
    shutil.copytree(ruta_, destino) if os.path.isdir(ruta_) else shutil.copy(ruta_, destino)
zip_path = shutil.make_archive("/content/salida_gtr_en_linea", "zip", carpeta)
print(sorted(os.listdir(carpeta + "/resultados")))
files.download(zip_path)